# 12 — 動画・カメラ入力

## 目標
- `cv2.VideoCapture` で動画ファイル / カメラを開く
- フレームを取得して処理を適用する
- `cv2.VideoWriter` で処理済み動画を保存する

## 注意
- Jupyter Notebook ではリアルタイムプレビューは困難。
  ここではフレームを静止画として取得し、matplotlib で確認します。
- カメラを使う場合はデバイス番号 (通常 `0`) を指定します。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import cv2
import numpy as np
from utils import DATA_DIR, show_nb

VIDEO_PATH = DATA_DIR / 'sample.mp4'
print(f'動画パス: {VIDEO_PATH}')
print(f'存在: {VIDEO_PATH.exists()}')

In [ ]:
if VIDEO_PATH.exists():
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f'解像度: {width}x{height}, FPS: {fps:.1f}, 総フレーム: {total}')
    cap.release()
else:
    print('[info] sample.mp4 が見つかりません。data/README.md を参照してください。')

In [ ]:
def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    return cv2.addWeighted(frame, 0.7, edges_bgr, 0.3, 0)

if VIDEO_PATH.exists():
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    n = 4
    step = max(1, total // n)
    pairs = []
    for i in range(n):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if ret:
            pairs.append((f'frame {i*step}', frame))
            pairs.append((f'processed {i*step}', process_frame(frame)))
    cap.release()
    show_nb(pairs, cols=2)
else:
    print('動画ファイルを data/ に配置してください。')

In [ ]:
# オプション: Lucas-Kanade オプティカルフロー (連続フレーム間)
if VIDEO_PATH.exists():
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    ret, frame1 = cap.read()
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY) if ret else None
    
    if gray1 is not None:
        # 特徴点
        p0 = cv2.goodFeaturesToTrack(gray1, 100, 0.3, 7)
        ret, frame2 = cap.read()
        if ret and p0 is not None:
            gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
            p1, st, _ = cv2.calcOpticalFlowPyrLK(gray1, gray2, p0, None)
            vis = frame2.copy()
            for new, old, s in zip(p1, p0, st):
                if s[0]:
                    a, b = new.ravel().astype(int)
                    c, d = old.ravel().astype(int)
                    cv2.arrowedLine(vis, (c,d), (a,b), (0,255,0), 2, tipLength=0.3)
            show_nb([('frame1', frame1), ('optical flow', vis)], cols=2)
    cap.release()